# Anhedonic AI — 72B Neuron Analysis
Complete analysis of all outputs: Original (geo+math), ASDiv, and MMLU (54 subjects) for Qwen2-VL-72B.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from matplotlib.patches import Patch

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

NUM_LAYERS   = 80
K_THRESHOLDS = [27, 30, 35, 40, 45, 50, 54]
LAYERS       = list(range(NUM_LAYERS))

BASE     = '/mnt/mahdipou/models/Anhedonic-AI/72-Exp1'
ORIG_DIR = f'{BASE}/neurons/orig'
ASDIV_DIR= f'{BASE}/ASDiv/neurons'
MMLU_DIR = f'{BASE}/MMLU/neurons'

SUBJECTS = [
    'abstract_algebra','anatomy','astronomy','business_ethics',
    'clinical_knowledge','college_biology','college_computer_science',
    'college_mathematics','college_medicine','computer_security',
    'conceptual_physics','econometrics','electrical_engineering',
    'elementary_mathematics','formal_logic','global_facts',
    'high_school_biology','high_school_chemistry','high_school_computer_science',
    'high_school_european_history','high_school_geography',
    'high_school_government_and_politics','high_school_macroeconomics',
    'high_school_mathematics','high_school_microeconomics','high_school_physics',
    'high_school_psychology','high_school_statistics','high_school_us_history',
    'high_school_world_history','human_aging','human_sexuality',
    'international_law','jurisprudence','logical_fallacies','machine_learning',
    'management','marketing','medical_genetics','miscellaneous','moral_disputes',
    'moral_scenarios','nutrition','philosophy','prehistory',
    'professional_accounting','professional_law','professional_medicine',
    'professional_psychology','public_relations','security_studies','sociology',
    'virology','world_religions'
]

print('Config ready. NUM_LAYERS=80')

In [ ]:
def load_set(path):
    df = pd.read_csv(path)
    return set(zip(df['layer'].tolist(), df['neuron'].tolist()))

def layer_dist(neuron_set):
    c = defaultdict(int)
    for (l, _) in neuron_set:
        c[l] += 1
    return [c.get(l, 0) for l in LAYERS]

def l_count(s, layer):
    return sum(1 for (l, _) in s if l == layer)

def band_count(s, l_start, l_end):
    return sum(1 for (l, _) in s if l_start <= l <= l_end)

# Orig
orig_reward = load_set(f'{ORIG_DIR}/universal_reward_neurons.csv')
orig_money  = load_set(f'{ORIG_DIR}/universal_money_neurons.csv')
orig_core   = load_set(f'{ORIG_DIR}/master_incentive_core.csv')

# ASDiv
asdiv_reward = load_set(f'{ASDIV_DIR}/asdiv_universal_reward_neurons.csv')
asdiv_money  = load_set(f'{ASDIV_DIR}/asdiv_universal_money_neurons.csv')
asdiv_core   = load_set(f'{ASDIV_DIR}/asdiv_master_incentive_core.csv')

# MMLU per-subject
print('Loading MMLU per-subject...')
mmlu_per_subject = {}
for subj in SUBJECTS:
    mmlu_per_subject[subj] = {
        'reward': load_set(f'{MMLU_DIR}/per_subject/{subj}/reward.csv'),
        'money':  load_set(f'{MMLU_DIR}/per_subject/{subj}/money.csv'),
        'core':   load_set(f'{MMLU_DIR}/per_subject/{subj}/core.csv'),
    }

# MMLU cross-subject
mmlu_k = {k: load_set(f'{MMLU_DIR}/cross_subject/k{k}.csv') for k in K_THRESHOLDS}
subject_counts_df = pd.read_csv(f'{MMLU_DIR}/cross_subject/subject_counts.csv')

# Key derived sets
shared_421 = mmlu_k[54] & asdiv_core
triple_49  = mmlu_k[54] & asdiv_core & orig_core

print(f'Orig   — reward:{len(orig_reward):>7}, money:{len(orig_money):>7}, core:{len(orig_core):>7}')
print(f'ASDiv  — reward:{len(asdiv_reward):>7}, money:{len(asdiv_money):>7}, core:{len(asdiv_core):>7}')
print(f'MMLU K=54:           {len(mmlu_k[54])}')
print(f'Shared 421 (K54 ∩ ASDiv):        {len(shared_421)}')
print(f'Triple 49  (K54 ∩ ASDiv ∩ Orig): {len(triple_49)}')

## 1 — Set sizes overview

In [ ]:
labels = [
    'Orig\nreward','Orig\nmoney','Orig\ncore',
    'ASDiv\nreward','ASDiv\nmoney','ASDiv\ncore',
    'MMLU\nK=27','MMLU\nK=30','MMLU\nK=35','MMLU\nK=40',
    'MMLU\nK=45','MMLU\nK=50','MMLU\nK=54',
    'Shared\n421','Triple\n49'
]
sizes = [
    len(orig_reward),len(orig_money),len(orig_core),
    len(asdiv_reward),len(asdiv_money),len(asdiv_core),
    *[len(mmlu_k[k]) for k in K_THRESHOLDS],
    len(shared_421),len(triple_49)
]
colors = ['#4878CF']*3 + ['#6ACC65']*3 + ['#D65F5F']*7 + ['#B47CC7','#C4AD66']

fig, ax = plt.subplots(figsize=(16,5))
bars = ax.bar(labels, sizes, color=colors, edgecolor='white', linewidth=0.5)
for bar, size in zip(bars, sizes):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.05,
            f'{size:,}', ha='center', va='bottom', fontsize=8)
ax.set_ylabel('Number of neurons')
ax.set_title('72B — Neuron set sizes across all experiments')
ax.set_yscale('log')
ax.legend(handles=[
    Patch(color='#4878CF', label='Original (geo+math)'),
    Patch(color='#6ACC65', label='ASDiv'),
    Patch(color='#D65F5F', label='MMLU K-thresholds'),
    Patch(color='#B47CC7', label='Shared 421 (K54 ∩ ASDiv)'),
    Patch(color='#C4AD66', label='Triple 49'),
])
plt.tight_layout()
plt.savefig('72b_set_sizes.png', bbox_inches='tight')
plt.show()

## 2 — Subject count distribution

In [ ]:
dist = subject_counts_df['subject_count'].value_counts().sort_index()

fig, axes = plt.subplots(1,2,figsize=(16,5))

axes[0].bar(dist.index, dist.values, color='#D65F5F', edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('Number of subjects neuron appears in')
axes[0].set_ylabel('Number of neurons')
axes[0].set_title('72B — Full distribution')
for k in K_THRESHOLDS:
    axes[0].axvline(k, color='navy', linestyle='--', linewidth=0.8, alpha=0.6)
    axes[0].text(k+0.2, axes[0].get_ylim()[1]*0.85, f'K={k}', fontsize=7, color='navy', rotation=90)

plateau = dist[dist.index >= 15]
axes[1].bar(plateau.index, plateau.values, color='#D65F5F', edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('Number of subjects')
axes[1].set_ylabel('Number of neurons')
axes[1].set_title('72B — Plateau region (subjects ≥ 15)')
for k in K_THRESHOLDS:
    if k >= 15:
        axes[1].axvline(k, color='navy', linestyle='--', linewidth=0.8, alpha=0.6)
        axes[1].text(k+0.2, axes[1].get_ylim()[1]*0.85, f'K={k}', fontsize=7, color='navy', rotation=90)

plt.tight_layout()
plt.savefig('72b_subject_count_dist.png', bbox_inches='tight')
plt.show()

## 3 — Layer distribution heatmap

In [ ]:
sets_ordered = {
    'Orig reward':  orig_reward,
    'Orig money':   orig_money,
    'Orig core':    orig_core,
    'ASDiv reward': asdiv_reward,
    'ASDiv money':  asdiv_money,
    'ASDiv core':   asdiv_core,
    'MMLU K=27':    mmlu_k[27],
    'MMLU K=35':    mmlu_k[35],
    'MMLU K=45':    mmlu_k[45],
    'MMLU K=54':    mmlu_k[54],
    'Shared 421':   shared_421,
    'Triple 49':    triple_49,
}

matrix_pct = []
for name, s in sets_ordered.items():
    d     = layer_dist(s)
    total = len(s) if s else 1
    matrix_pct.append([c/total*100 for c in d])

df_heatmap = pd.DataFrame(matrix_pct,
                           index=list(sets_ordered.keys()),
                           columns=[f'L{l}' for l in LAYERS])

fig, ax = plt.subplots(figsize=(28,7))
sns.heatmap(df_heatmap, ax=ax, cmap='YlOrRd', annot=False,
            linewidths=0.2, linecolor='white',
            cbar_kws={'label':'% of set in layer'})
ax.set_title('72B — Layer distribution (% of each set) across all neuron sets')
ax.set_xlabel('Layer')
plt.tight_layout()
plt.savefig('72b_layer_heatmap.png', bbox_inches='tight')
plt.show()

## 4 — Layer distributions: absolute counts

In [ ]:
plot_sets = {
    'Orig reward':  (orig_reward,  '#4878CF'),
    'Orig money':   (orig_money,   '#4878CF'),
    'Orig core':    (orig_core,    '#2255AA'),
    'ASDiv reward': (asdiv_reward, '#6ACC65'),
    'ASDiv money':  (asdiv_money,  '#6ACC65'),
    'ASDiv core':   (asdiv_core,   '#2A9A2A'),
    'MMLU K=27':    (mmlu_k[27],   '#D65F5F'),
    'MMLU K=40':    (mmlu_k[40],   '#D65F5F'),
    'MMLU K=54':    (mmlu_k[54],   '#AA2222'),
    'Shared 421':   (shared_421,   '#B47CC7'),
    'Triple 49':    (triple_49,    '#C4AD66'),
}

fig, axes = plt.subplots(3,4,figsize=(24,14), sharey=False)
axes = axes.flatten()

# Proportional reward band for 72B: L23-40 (~29-50% of 80 layers)
BAND_START, BAND_END, PEAK = 23, 40, 38

for ax, (name, (s, color)) in zip(axes, plot_sets.items()):
    d = layer_dist(s)
    ax.bar(LAYERS, d, color=color, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{name} (n={len(s):,})', fontsize=9)
    ax.set_xlabel('Layer', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.axvspan(BAND_START-0.5, BAND_END+0.5, alpha=0.08, color='red')
    ax.axvline(PEAK, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.tick_params(labelsize=7)

for ax in axes[len(plot_sets):]:
    ax.set_visible(False)

fig.suptitle('72B — Layer distributions (red band = L23-40, dashed = L38)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('72b_layer_distributions.png', bbox_inches='tight')
plt.show()

## 5 — Pairwise Jaccard similarity matrix

In [ ]:
jaccard_sets = {
    'Orig core':  orig_core,
    'ASDiv core': asdiv_core,
    'MMLU K=27':  mmlu_k[27],
    'MMLU K=30':  mmlu_k[30],
    'MMLU K=35':  mmlu_k[35],
    'MMLU K=40':  mmlu_k[40],
    'MMLU K=45':  mmlu_k[45],
    'MMLU K=50':  mmlu_k[50],
    'MMLU K=54':  mmlu_k[54],
    'Shared 421': shared_421,
    'Triple 49':  triple_49,
}

names = list(jaccard_sets.keys())
n = len(names)
jmat = np.zeros((n,n))
for i,(na,sa) in enumerate(jaccard_sets.items()):
    for j,(nb,sb) in enumerate(jaccard_sets.items()):
        u = len(sa|sb)
        jmat[i,j] = len(sa&sb)/u if u else 0.0

fig, ax = plt.subplots(figsize=(12,10))
sns.heatmap(jmat, ax=ax, annot=True, fmt='.3f',
            xticklabels=names, yticklabels=names,
            cmap='Blues', vmin=0, vmax=1,
            linewidths=0.5, linecolor='white')
ax.set_title('72B — Pairwise Jaccard similarity')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('72b_jaccard_matrix.png', bbox_inches='tight')
plt.show()

## 6 — MMLU per-subject core sizes

In [ ]:
subj_core_sizes = {s: len(mmlu_per_subject[s]['core']) for s in SUBJECTS}
sorted_subjects = sorted(subj_core_sizes.items(), key=lambda x: -x[1])
s_labels = [s[0].replace('_','\n') for s in sorted_subjects]
s_cores  = [s[1] for s in sorted_subjects]

fig, ax = plt.subplots(figsize=(22,6))
ax.bar(range(len(s_labels)), s_cores, color='#D65F5F', edgecolor='white', linewidth=0.3)
ax.axhline(np.mean(s_cores), color='navy', linestyle='--', linewidth=1.2, label=f'Mean={np.mean(s_cores):.0f}')
ax.set_xticks(range(len(s_labels)))
ax.set_xticklabels(s_labels, fontsize=6, rotation=90)
ax.set_ylabel('Core neurons')
ax.set_title('72B MMLU — Per-subject core sizes (sorted descending)')
ax.legend()
plt.tight_layout()
plt.savefig('72b_per_subject_cores.png', bbox_inches='tight')
plt.show()

print(f'Mean: {np.mean(s_cores):.0f}, Std: {np.std(s_cores):.0f}')
print(f'Min: {sorted_subjects[-1]}')
print(f'Max: {sorted_subjects[0]}')

## 7 — Pair plots: reward / money / core at all K values

In [ ]:
# Compute MMLU reward/money at each K from per-subject data
reward_counts = defaultdict(int)
money_counts  = defaultdict(int)
core_counts   = defaultdict(int)

for subj in SUBJECTS:
    for n in mmlu_per_subject[subj]['reward']: reward_counts[n] += 1
    for n in mmlu_per_subject[subj]['money']:  money_counts[n]  += 1
    for n in mmlu_per_subject[subj]['core']:   core_counts[n]   += 1

mmlu_k_reward = {k: {n for n,c in reward_counts.items() if c>=k} for k in K_THRESHOLDS}
mmlu_k_money  = {k: {n for n,c in money_counts.items()  if c>=k} for k in K_THRESHOLDS}
mmlu_k_core   = {k: {n for n,c in core_counts.items()   if c>=k} for k in K_THRESHOLDS}

COLORS = {'orig':'#4878CF','asdiv':'#6ACC65','mmlu':'#D65F5F'}
BAND_START, BAND_END = 23, 40

def plot_group(k, ax_r, ax_m, ax_c):
    sets = [
        ('Orig',        orig_reward,       orig_money,       orig_core,       COLORS['orig']),
        ('ASDiv',       asdiv_reward,      asdiv_money,      asdiv_core,      COLORS['asdiv']),
        (f'MMLU K={k}', mmlu_k_reward[k],  mmlu_k_money[k],  mmlu_k_core[k],  COLORS['mmlu']),
    ]
    x   = np.array(LAYERS)
    w   = 0.26
    off = [-w, 0, w]
    for ax, cidx, title in [(ax_r,1,'Reward neurons'),(ax_m,2,'Money neurons'),(ax_c,3,'Core neurons')]:
        for (label,sr,sm,sc,color), offset in zip(sets, off):
            s      = [sr,sm,sc][cidx-1]
            counts = layer_dist(s)
            ax.bar(x+offset, counts, w, label=f'{label} (n={len(s):,})',
                   color=color, edgecolor='white', linewidth=0.3, alpha=0.9)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xlabel('Layer', fontsize=9)
        ax.set_ylabel('Neuron count', fontsize=9)
        ax.set_xticks(LAYERS[::4])
        ax.set_xticklabels(LAYERS[::4], fontsize=7)
        ax.axvspan(BAND_START-0.5, BAND_END+0.5, alpha=0.07, color='red')
        ax.axvline(38, color='red',  linestyle='--', linewidth=0.8, alpha=0.5)
        ax.legend(fontsize=8, loc='upper right')
        ax.tick_params(axis='y', labelsize=8)

print('K-threshold reward/money/core sets computed.')
print(f'  K    Reward    Money    Core')
for k in K_THRESHOLDS:
    print(f'  {k:<4} {len(mmlu_k_reward[k]):>7}  {len(mmlu_k_money[k]):>7}  {len(mmlu_k_core[k]):>7}')

In [ ]:
for k in K_THRESHOLDS:
    fig, axes = plt.subplots(1,3,figsize=(22,5))
    fig.suptitle(
        f'72B  K={k}  |  Orig vs ASDiv vs MMLU\n'
        f'Red band = L23-40  |  dashed red = L38',
        fontsize=11, y=1.02
    )
    plot_group(k, axes[0], axes[1], axes[2])
    plt.tight_layout()
    plt.savefig(f'72b_comparison_k{k}.png', bbox_inches='tight', dpi=130)
    plt.show()
    print(f'K={k} saved → 72b_comparison_k{k}.png')

## 8 — Recall / precision vs orig and ASDiv

In [ ]:
comparison_sets = {
    'MMLU K=27': mmlu_k[27], 'MMLU K=30': mmlu_k[30],
    'MMLU K=35': mmlu_k[35], 'MMLU K=40': mmlu_k[40],
    'MMLU K=45': mmlu_k[45], 'MMLU K=50': mmlu_k[50],
    'MMLU K=54': mmlu_k[54], 'Shared 421': shared_421, 'Triple 49': triple_49,
}

fig, axes = plt.subplots(1,2,figsize=(16,6))
for ax, (ref_name, ref_set) in zip(axes, [('Orig core', orig_core), ('ASDiv core', asdiv_core)]):
    recalls, precisions, labs = [], [], []
    for cname, cset in comparison_sets.items():
        inter = ref_set & cset
        recalls.append(len(inter)/len(ref_set)*100 if ref_set else 0)
        precisions.append(len(inter)/len(cset)*100 if cset else 0)
        labs.append(cname)
    x, w = np.arange(len(labs)), 0.35
    ax.bar(x-w/2, recalls,    w, label='Recall (% of ref recovered)', color='#4878CF', edgecolor='white')
    ax.bar(x+w/2, precisions, w, label='Precision (% of set in ref)',  color='#D65F5F', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(labs, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('%')
    ax.set_title(f'72B — Recall & precision vs {ref_name}')
    ax.legend(fontsize=9)
    ax.set_ylim(0,105)
    for xi,(r,p) in enumerate(zip(recalls,precisions)):
        ax.text(xi-w/2, r+1, f'{r:.0f}%', ha='center', fontsize=7)
        ax.text(xi+w/2, p+1, f'{p:.0f}%', ha='center', fontsize=7)

plt.tight_layout()
plt.savefig('72b_recall_precision.png', bbox_inches='tight')
plt.show()

## 9 — Ablation candidate summary

In [ ]:
candidates = {
    'Orig core':                   orig_core,
    'ASDiv core':                  asdiv_core,
    'MMLU K=50':                   mmlu_k[50],
    'MMLU K=54':                   mmlu_k[54],
    'Shared 421 (K54 ∩ ASDiv)':    shared_421,
    'Triple 49  (K54 ∩ ASDiv ∩ Orig)': triple_49,
}

print(f'{"Set":<40} {"Size":>6}  {"L23-40":>7}  {"L38-43":>7}  {"L78-79":>7}  {"% in L23-40":>12}')
print('-'*82)
for name, s in candidates.items():
    size  = len(s)
    b2340 = band_count(s, 23, 40)
    b3843 = band_count(s, 38, 43)
    b7879 = band_count(s, 78, 79)
    pct   = b2340/size*100 if size else 0
    print(f'{name:<40} {size:>6}  {b2340:>7}  {b3843:>7}  {b7879:>7}  {pct:>11.1f}%')

fig, axes = plt.subplots(1,2,figsize=(16,5))
for ax,(name,s) in zip(axes,[('Shared 421 (K=54 ∩ ASDiv)', shared_421),
                               ('Triple 49 (K=54 ∩ ASDiv ∩ Orig)', triple_49)]):
    d = layer_dist(s)
    colors_bar = ['#D65F5F' if 23<=l<=40 else '#AAAAAA' for l in LAYERS]
    ax.bar(LAYERS, d, color=colors_bar, edgecolor='white', linewidth=0.3)
    ax.set_title(name)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Neuron count')
    ax.axvline(38, color='darkred', linestyle='--', linewidth=1, label='L38')
    ax.legend(fontsize=9)

plt.suptitle('72B Ablation candidates — layer profiles (red = L23-40 band)', fontsize=12)
plt.tight_layout()
plt.savefig('72b_ablation_candidates.png', bbox_inches='tight')
plt.show()

## 10 — Export ablation JSON files

In [ ]:
import json

def export_ablation_json(neuron_set, path):
    d = defaultdict(list)
    for (layer, neuron) in sorted(neuron_set):
        d[str(layer)].append(neuron)
    with open(path, 'w') as f:
        json.dump(dict(d), f, indent=2)
    print(f'Saved {path}  ({len(neuron_set)} neurons)')

out_dir = f'{BASE}/ablation_sets'
os.makedirs(out_dir, exist_ok=True)

export_ablation_json(shared_421, f'{out_dir}/neurons_shared421.json')
export_ablation_json(triple_49,  f'{out_dir}/neurons_triple49.json')
export_ablation_json(mmlu_k[54], f'{out_dir}/neurons_mmlu_k54.json')
export_ablation_json(mmlu_k[50], f'{out_dir}/neurons_mmlu_k50.json')
export_ablation_json(orig_core,  f'{out_dir}/neurons_orig_core.json')

print('\nAll 72B ablation JSON files exported.')